In [ ]:
import msgpack, msgpack_numpy, numpy as np, requests
from droid_plus.policies.image_tools import resize_with_pad

POLICY_URL = "http://127.0.0.1:8000"     # your SSH tunnel endpoint
PROMPT     = "lift the white color object"

print("health:", requests.get(f"{POLICY_URL}/health", timeout=10).json())

def infer(left, wrist, q, grip, prompt=PROMPT):
    req = {
        "images": {
            "left":  resize_with_pad(np.asarray(left,  np.uint8), 224, 224),
            "wrist": resize_with_pad(np.asarray(wrist, np.uint8), 224, 224),
        },
        "state":  np.concatenate([np.asarray(q, np.float32).reshape(-1), np.float32([grip])]),
        "prompt": prompt,
    }
    blob = msgpack.packb(req, default=msgpack_numpy.encode, use_bin_type=True)
    r = requests.post(f"{POLICY_URL}/infer", data=blob,
                      headers={"Content-Type": "application/msgpack"}, timeout=20)
    r.raise_for_status()
    out = msgpack.unpackb(r.content, object_hook=msgpack_numpy.decode, raw=False)
    return np.asarray(out["actions"], dtype=float)     # (10, 8)


health: {'ok': True, 'checkpoint': '/home/developer/lerobot_ft/outputs/lift_white_color_object_teleop_20260909_190957/checkpoints/last/pretrained_model/', 'device': 'cuda', 'action_dim': 8}


In [ ]:
from droid_plus.robot import DroidPlus
droid = DroidPlus()
g = droid.gripper

left  = droid.get_left_image(jpeg_quality=90)         # RGB uint8
wrist = droid.get_wrist_image(jpeg_quality=90)
q     = np.asarray(droid.get_current_joint_state()["positions"], np.float32)
try:
    grip = float(g.gripper_position_frac())           # 0=open, 1=closed
except Exception:
    grip = 0.0

print("current q :", np.round(q, 3), " grip:", round(grip, 3))

chunk = infer(left, wrist, q, grip)
# print("chunk shape:", chunk.shape)
# print("action[0]  :", np.round(chunk[0], 3))
# print("|action[0][:7] - q| :", np.round(np.abs(chunk[0,:7] - q), 3), " max:", round(np.abs(chunk[0,:7]-q).max(), 3))
# print("gripper col:", np.round(chunk[:,7], 2))
# print("per-step Δ (rad):", np.round(np.abs(np.diff(chunk[:,:7], axis=0)).max(axis=1), 3))


gripper initialized None
current q : [ 2.000e-03 -7.840e-01  2.000e-03 -2.356e+00  5.000e-03  1.577e+00
  7.930e-01]  grip: 0.0


In [ ]:
g.open()

{'ok': True, 'position': 0, 'object_detected': False, 'accepted': False}

In [79]:
import time

CONTROL_HZ    = 15.0
EXECUTE_STEPS = 10          # run ~1 s of the chunk, then re-infer
MAX_CHUNKS    = 36
MAX_STEP      = 0.05        # rad/tick clamp toward each target (absorbs the q->action[0] gap)
STEP_ABORT    = 0.5        # abort a chunk whose own steps exceed this
DT = 1.0 / CONTROL_HZ

Q_MIN = np.array([-2.7437,-1.7837,-2.9007,-3.0421,-2.8065, 0.5445,-3.0159])
Q_MAX = np.array([ 2.7437, 1.7837, 2.9007,-0.1518, 2.8065, 4.5169, 3.0159])
MARGIN = 0.12

GRIP_SPEED, GRIP_FORCE = 120, 80
APPROVE_FIRST_CLOSE = True
DRY_RUN = False             # print only; no arm/gripper commands

def get_obs():
    left  = droid.get_left_image(jpeg_quality=90)
    wrist = droid.get_wrist_image(jpeg_quality=90)
    q     = np.asarray(droid.get_current_joint_state()["positions"], float)
    try:    grip = float(g.gripper_position_frac())
    except Exception: grip = 0.0
    return left, wrist, q, grip

def send_q(qc, seq):
    if not DRY_RUN:
        droid.set_target_joint_state(np.clip(qc, Q_MIN, Q_MAX), velocities=[0.0]*7, seq=seq)

def soft_stop():
    time.sleep(0.15)                    # let franky's 100 ms watchdog decay first
    if not DRY_RUN:
        try: droid.stop()
        except Exception: pass


In [82]:
# ============================================================================
#  TABLE-COLLISION SAFETY  -  keep the gripper >= TABLE_CLEARANCE above the table
# ============================================================================
# Dependency-free analytic forward kinematics for the Franka FR3 (modified DH).
# We track the base-frame height (Z) of the gripper tip and refuse any commanded
# joint step that would drive it below  table_surface + TABLE_CLEARANCE.
# Verified: q=[0,-0.785,0,-2.356,0,1.571,0.785] -> panda_link8 = [0.307, 0, 0.590].

TABLE_CLEARANCE = 0.015     # 1.5 cm hard floor above the table surface
TCP_OFFSET_Z    = 0.16      # metres from the flange (panda_link8) to the gripper
                            # tip, along the flange local +Z. Robotiq 2F-85 on the
                            # Franka mount ~= 0.15-0.17 m. The calibration cell
                            # below makes the exact value non-critical (the same
                            # offset is used at calibration and at run time).

_FR3_DH = [                 # (a, d, alpha);  theta = q_i  (last row = fixed flange)
    (0.0,     0.333,  0.0),
    (0.0,     0.0,   -np.pi / 2),
    (0.0,     0.316,  np.pi / 2),
    (0.0825,  0.0,    np.pi / 2),
    (-0.0825, 0.384, -np.pi / 2),
    (0.0,     0.0,    np.pi / 2),
    (0.088,   0.0,    np.pi / 2),
    (0.0,     0.107,  0.0),
]

def _fr3_flange_T(q):
    """4x4 base->flange (panda_link8) transform for arm joints q (7,)."""
    q = np.asarray(q, float).reshape(-1)[:7]
    T = np.eye(4)
    for (a, d, alpha), th in zip(_FR3_DH, list(q) + [0.0]):
        ca, sa, ct, st = np.cos(alpha), np.sin(alpha), np.cos(th), np.sin(th)
        T = T @ np.array([
            [ct,    -st,    0.0,  a],
            [st*ca,  ct*ca, -sa, -sa*d],
            [st*sa,  ct*sa,  ca,  ca*d],
            [0.0,    0.0,    0.0,  1.0],
        ])
    return T

def gripper_z(q):
    """Base-frame height (m) of the gripper tip for arm joint vector q (7,)."""
    T = _fr3_flange_T(q)
    return float(T[2, 3] + T[2, 2] * TCP_OFFSET_Z)   # flange origin + Z_flange * offset

# --- table height in this FK frame ------------------------------------------
# Either set TABLE_Z directly (if you measured the tip height in base frame) or
# run the calibration cell below.


def z_floor():
    if TABLE_Z is None:
        raise RuntimeError("TABLE_Z is not set - run the table-height calibration cell")
    return TABLE_Z + TABLE_CLEARANCE


In [83]:
# --- CALIBRATE the table height -------------------------------------------------
# Jog the arm (teleop) until the gripper tip just rests on the table surface,
# then run this cell ONCE. Re-run whenever the gripper/mount changes.
#
# Alternative without touching the table: hold the tip a known height H above the
# table, run the two lines, then do  TABLE_Z = TABLE_Z - H.

# TABLE_Z = None
# _qcal = get_obs()[2]
# TABLE_Z = gripper_z(_qcal)
# print(f"calibrated TABLE_Z = {TABLE_Z:+.4f} m   from q = {np.round(_qcal, 3)}")
# print(f"safety floor       = {z_floor():+.4f} m   (= table + {TABLE_CLEARANCE*100:.1f} cm)")
# print(f"gripper tip now     = {gripper_z(get_obs()[2]):+.4f} m")


In [84]:
TABLE_Z = -0.0490

In [320]:
from pathlib import Path
from PIL import Image
from datetime import datetime

INFERENCE_ROOT = Path("inference_outputs")

# --- set the object position before each new position -----------------------
# Update POSITION whenever you move the object to a new spot, then re-run this
# cell once. Every run below (1 healthy + 7 joint-failure) reuses it and gets
# its own timestamped subfolder, so nothing gets overwritten.
POSITION = "pos_10"
print("POSITION set to:", POSITION)


POSITION set to: pos_10


In [347]:
if not DRY_RUN:
    g.open(speed=GRIP_SPEED)
    droid.robot.set_command_timeout(3.0)

q0 = get_obs()[2]
START_Q = np.array([0.,0.,0.,-1.571,0.,1.571,0.])   # your teleop start pose == franky HOME
print("current q:", np.round(q0,3), " -> START_Q gap:", round(np.abs(START_Q-q0).max(),3))
input("workspace clear, E-stop in hand — Enter to reset to start ")

sent = q0.astype(float); seq = 0
try:
    while np.abs(START_Q - sent).max() > 1e-3:
        sent = np.clip(sent + np.clip(START_Q - sent, -0.03, 0.03), Q_MIN, Q_MAX)
        send_q(sent, seq); seq += 1; time.sleep(DT)
    for _ in range(8): send_q(sent, seq); seq += 1; time.sleep(DT)
finally:
    soft_stop()
print("at start:", np.round(get_obs()[2], 3))



current q: [-0.296 -0.003 -0.078 -2.542 -0.141  1.571  0.481]  -> START_Q gap: 0.971
at start: [-0.     0.    -0.    -1.572  0.     1.571  0.   ]


In [ ]:
import random

FAIL_JOINT = int(input("joint index to simulate failure (0-6, blank/-1 for none): ") or -1)
if 0 <= FAIL_JOINT < 7:
    FAIL_STEP = random.randint(6, 60)   # control step (global, across chunks) where the fault kicks in
    print(f"simulating a stuck joint {FAIL_JOINT}: it will freeze at whatever value it last had, "
          f"starting at control step {FAIL_STEP} (randomly chosen in [6, 60])")
else:
    FAIL_JOINT = -1
    FAIL_STEP = None
FAIL_STEP_ACTUAL = None   # step at which the freeze actually engaged (None if never reached)

RUN_LABEL = "healthy" if FAIL_JOINT == -1 else f"joint_{FAIL_JOINT}_failure"
RUN_DIR = INFERENCE_ROOT / str(POSITION) / f"{RUN_LABEL}_{datetime.now():%Y%m%d_%H%M%S}"
LEFT_DIR, WRIST_DIR = RUN_DIR / "left", RUN_DIR / "wrist"
LEFT_DIR.mkdir(parents=True, exist_ok=True)
WRIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"saving camera frames under: {RUN_DIR}  (left/ and wrist/ subfolders, ready for ffmpeg)")

LIFT_HEIGHT = 0.10          # end the episode once the gripper tip has risen this far above
                             # the height it was at when it first closed (i.e. after a pick
                             # attempt) - moving around above this height before ever closing
                             # does NOT end the episode (that's just the reach/approach phase)
SAVE_EVERY_N_STEPS = 1       # only grab a frame on every Nth control tick within a chunk

frame_idx = 0

def save_frame(left_img, wrist_img):
    global frame_idx
    frame_idx += 1
    Image.fromarray(left_img).save(LEFT_DIR / f"frame_{frame_idx:04d}.jpg", quality=90)
    Image.fromarray(wrist_img).save(WRIST_DIR / f"frame_{frame_idx:04d}.jpg", quality=90)

class EpisodeEnd(Exception):
    pass

_grip = {"closed": None}
close_z = None   # gripper-tip height (m) at the moment it first closed; set once
seq = 1000
step_count = 0
Z_FLOOR = z_floor()   # raises if the table-height calibration cell hasn't run
print(f"table-collision guard active: gripper tip must stay >= {Z_FLOOR:+.4f} m")

EPISODE_STATUS = None       # "lifted" | "max_chunks" | "interrupted" | "aborted"
ABORT_REASON   = None
CHUNKS_RUN     = 0
LIFT_ACHIEVED  = False
episode_start = time.time()
try:
    for ci in range(1, MAX_CHUNKS + 1):
        CHUNKS_RUN = ci
        left, wrist, q, grip = get_obs()
        save_frame(left, wrist)
        chunk = infer(left, wrist, q, grip)

        raw = chunk[:EXECUTE_STEPS, :7]
        tgt = np.clip(raw, Q_MIN + MARGIN, Q_MAX - MARGIN)
        gplan = chunk[:EXECUTE_STEPS, 7]

        step_max = np.abs(np.diff(np.vstack([q, tgt]), axis=0)).max()
        if step_max > STEP_ABORT:
            raise RuntimeError(f"chunk step {step_max:.3f} rad > STEP_ABORT")
        if not np.allclose(raw, tgt, atol=1e-6):
            print(f"   (clipped {int((raw!=tgt).sum())} target values to joint limits)")

        # --- table-collision guard: lowest gripper tip over the planned chunk ---
        z_plan = np.array([gripper_z(tt) for tt in tgt])
        z_now  = gripper_z(q)
        if z_plan.min() < Z_FLOOR:
            raise RuntimeError(
                f"TABLE SAFETY: chunk {ci} would lower the gripper tip to "
                f"z={z_plan.min():+.3f} m < floor {Z_FLOOR:+.3f} m - stopping")

        close = bool(gplan.max() > 0.5)
        print(f"[{ci:02d}/{MAX_CHUNKS}] first_gap={np.abs(tgt[0]-q).max():.3f} "
              f"reach={np.abs(tgt[-1]-q).max():.3f} grip={gplan.min():.2f}..{gplan.max():.2f} "
              f"close={close} z_now={z_now:+.3f} z_plan_min={z_plan.min():+.3f}"
              + ("  [DRY]" if DRY_RUN else ""))

        if not DRY_RUN:
            if close and _grip["closed"] is not True:
                if APPROVE_FIRST_CLOSE and _grip["closed"] is None:
                    input(f"[chunk {ci}] policy wants CLOSE - Enter to allow, interrupt to abort ")
                g.close_async(speed=GRIP_SPEED, force=GRIP_FORCE); _grip["closed"] = True
                if close_z is None:
                    close_z = z_now
                    print(f"   gripper closed at z={close_z:+.3f} m - lift-height check now armed")
            elif not close and _grip["closed"] is not False:
                g.open_async(speed=GRIP_SPEED); _grip["closed"] = False

        sent = q.copy(); t = time.time()
        for k in range(EXECUTE_STEPS):
            step_count += 1
            cand = np.clip(sent + np.clip(tgt[k] - sent, -MAX_STEP, MAX_STEP), Q_MIN, Q_MAX)
            if FAIL_JOINT != -1 and step_count >= FAIL_STEP:
                # simulated joint failure, engaged at the randomly-chosen step: never accept
                # the policy's value for this joint from here on - hold it at whatever was
                # last sent, before it goes to franky.
                if FAIL_STEP_ACTUAL is None:
                    FAIL_STEP_ACTUAL = step_count
                    print(f"   >>> joint {FAIL_JOINT} failure engaged at step {step_count}")
                cand[FAIL_JOINT] = sent[FAIL_JOINT]
            z_cand = gripper_z(cand)
            if z_cand < Z_FLOOR:
                raise RuntimeError(
                    f"TABLE SAFETY: chunk {ci} step {k} would put the gripper tip at "
                    f"z={z_cand:+.3f} m < floor {Z_FLOOR:+.3f} m - stopping")
            sent = cand
            send_q(sent, seq); seq += 1
            if step_count % SAVE_EVERY_N_STEPS == 0:
                save_frame(droid.get_left_image(jpeg_quality=90), droid.get_wrist_image(jpeg_quality=90))

            if close_z is not None and (z_cand - close_z) >= LIFT_HEIGHT:
                save_frame(droid.get_left_image(jpeg_quality=90), droid.get_wrist_image(jpeg_quality=90))
                LIFT_ACHIEVED = True
                raise EpisodeEnd(
                    f"gripper tip lifted {(z_cand - close_z)*100:.1f} cm above its close height "
                    f"(z_close={close_z:+.3f} m, gripper_closed={_grip['closed']}) - ending episode")

            t += DT; time.sleep(max(0.0, t - time.time()))
    else:
        EPISODE_STATUS = "max_chunks"
        print("MAX_CHUNKS reached")
except EpisodeEnd as e:
    save_frame(droid.get_left_image(jpeg_quality=90), droid.get_wrist_image(jpeg_quality=90))
    EPISODE_STATUS = "lifted"
    print("episode end:", e)
except KeyboardInterrupt:
    EPISODE_STATUS = "interrupted"
    print("interrupted")
except Exception as e:
    EPISODE_STATUS = "aborted"
    ABORT_REASON = f"{type(e).__name__}: {e}"
    print("ABORT:", ABORT_REASON)
finally:
    save_frame(droid.get_left_image(jpeg_quality=90), droid.get_wrist_image(jpeg_quality=90))
    soft_stop()
    EPISODE_DURATION = time.time() - episode_start
    GRIPPER_CLOSED_AT_END = bool(_grip["closed"])
    FRAMES_SAVED = frame_idx
    print("final q:", np.round(get_obs()[2], 3))
    print(f"saved {frame_idx} left+wrist frame pairs to {RUN_DIR}")
    print(f"episode status={EPISODE_STATUS} duration={EPISODE_DURATION:.1f}s "
          f"lift_achieved={LIFT_ACHIEVED} gripper_closed={GRIPPER_CLOSED_AT_END}")
    if FAIL_JOINT != -1:
        print(f"fail_step (planned)={FAIL_STEP}  fail_step_actual (engaged)={FAIL_STEP_ACTUAL}")


In [ ]:
# --- record this episode's outcome to a CSV --------------------------------
# Run this right after the inference cell above finishes (success or abort).
import csv, json

METRICS_CSV = INFERENCE_ROOT / "metrics.csv"
METRICS_FIELDS = [
    "timestamp", "position", "run_label", "fail_joint", "fail_step", "fail_step_actual",
    "outcome", "task_status", "task_status_desc",
    "episode_status", "abort_reason", "duration_sec", "chunks_run",
    "frames_saved", "lift_achieved", "gripper_closed",
    "final_joint_state", "final_gripper_xyz", "run_dir",
]

TASK_STATUS_OPTIONS = {
    "a": "Grasped and lifted the object",
    "b": "Grasped but failed to lift",
    "c": "Reached object location but failed to grasp",
    "d": "Failed to reach the object location",
}

# --- migrate an older metrics.csv (missing the new columns) in place --------
if METRICS_CSV.exists():
    with open(METRICS_CSV, newline="") as f:
        existing_header = next(csv.reader(f), [])
    if existing_header and existing_header != METRICS_FIELDS:
        with open(METRICS_CSV, newline="") as f:
            existing_rows = list(csv.DictReader(f))
        with open(METRICS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=METRICS_FIELDS)
            w.writeheader()
            for r in existing_rows:
                w.writerow({k: r.get(k, "") for k in METRICS_FIELDS})
        print(f"migrated {METRICS_CSV} to the new column set")

outcome = ""
while outcome not in ("success", "failure"):
    outcome = input("episode outcome - success or failure? ").strip().lower()

task_status = ""
while task_status not in TASK_STATUS_OPTIONS:
    print("task status options:")
    for k, v in TASK_STATUS_OPTIONS.items():
        print(f"  {k} - {v}")
    task_status = input("select task status (a/b/c/d): ").strip().lower()

# gripper tip Cartesian coordinate at end of episode - reuses the FR3 FK
# (_fr3_flange_T, TCP_OFFSET_Z) already defined in the table-safety cell.
def gripper_xyz(q):
    T = _fr3_flange_T(q)
    return T[:3, 3] + T[:3, 2] * TCP_OFFSET_Z

final_q = get_obs()[2]
final_xyz = gripper_xyz(final_q)

row = {
    "timestamp":         datetime.now().isoformat(timespec="seconds"),
    "position":          POSITION,
    "run_label":         RUN_LABEL,
    "fail_joint":        FAIL_JOINT,
    "fail_step":         FAIL_STEP if FAIL_STEP is not None else "",
    "fail_step_actual":  FAIL_STEP_ACTUAL if FAIL_STEP_ACTUAL is not None else "",
    "outcome":           outcome,
    "task_status":       task_status,
    "task_status_desc":  TASK_STATUS_OPTIONS[task_status],
    "episode_status":    EPISODE_STATUS,
    "abort_reason":      ABORT_REASON or "",
    "duration_sec":      round(EPISODE_DURATION, 2),
    "chunks_run":        CHUNKS_RUN,
    "frames_saved":      FRAMES_SAVED,
    "lift_achieved":     LIFT_ACHIEVED,
    "gripper_closed":    GRIPPER_CLOSED_AT_END,
    "final_joint_state": json.dumps([round(float(v), 4) for v in final_q]),
    "final_gripper_xyz": json.dumps([round(float(v), 4) for v in final_xyz]),
    "run_dir":           str(RUN_DIR),
}

write_header = not METRICS_CSV.exists()
with open(METRICS_CSV, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=METRICS_FIELDS)
    if write_header:
        w.writeheader()
    w.writerow(row)

print(f"logged to {METRICS_CSV}:")
for k, v in row.items():
    print(f"  {k}: {v}")
